In [1]:
import sys
sys.path.append('../../Simulate/')

from BSReadSim import BSReadSim

In [2]:
import os
import random
import numpy as np
import subprocess

from Bio import SeqIO
from tqdm import tqdm
from scipy.stats import bernoulli
from typing import Dict, Union, Tuple
from threading import Lock
from concurrent.futures import ThreadPoolExecutor, as_completed

from LockedIterator import LockedIterator
from SetMethylation import SetMethylation
from StreamReads import StreamReads
from StreamHTSIM import StreamHTSIM
from UtilityFunctions import get_htsim_path
from ParseGenome import ParseGenome


In [3]:
working_path = "/home/wbguo/iproject/BSReadSim/test/"

ref_fasta = working_path + "data/ref/BSB_test.fa"
outdir = working_path + "outdir"

In [4]:
self = BSReadSim(ref_fasta=ref_fasta, outdir=outdir, overwrite_db=True,
                     meth_db_path="/home/wbguo/iproject/BSReadSim/test/outdir/",
                     n_threads=1, num_reads=100000, 
                     verbose =True, shuffle=False, gzip=False)

Initiating genome...
Initiating methylation profile...

[Initiating meth_db] for chr10...

[Initiating meth_db] for chr11...

[Initiating meth_db] for chr12...

[Initiating meth_db] for chr13...

[Initiating meth_db] for chr14...

[Initiating meth_db] for chr15...


../../Simulate/StreamReads.py:38: UserWarning: Fastq file exists, will overwrite... /home/wbguo/iproject/BSReadSim/test/outdir/sim_1.fastq
  warnings.warn(f'Fastq file exists, will overwrite... {fastq_file}')
../../Simulate/StreamReads.py:38: UserWarning: Fastq file exists, will overwrite... /home/wbguo/iproject/BSReadSim/test/outdir/sim_2.fastq
  warnings.warn(f'Fastq file exists, will overwrite... {fastq_file}')


In [5]:
print('Simulating methylated Reads...')
print(f'[CMD]: {" ".join(self.cmd_part)}')
if self.verbose:
    print(f'[INFO]: #reads/#read pairs for each contig', end=" ")
    print(self.count_dict)

Simulating methylated Reads...
[CMD]: /home/wbguo/iproject/BSReadSim/HTSIM/htsim /home/wbguo/iproject/BSReadSim/test/data/ref/BSB_test.fa -i 400 -I 25 -m 100 -M 1000 -1 100 -2 100 -e 0 -A 0.05 -u 1 -f 1 -g None -r 0.001 -R 0.15 -X 0.15 -h 0 -s -1 -T 0 -x None -b None -B None -D None
[INFO]: #reads/#read pairs for each contig {'chr10': 10794, 'chr11': 10808, 'chr12': 10749, 'chr13': 8569, 'chr14': 8953, 'chr15': 127}


In [6]:
for contig_id in self.count_dict.keys():
    if self.count_dict[contig_id] == 0: # can be 0, need to test if reads < self.n_threads TODO
        continue
    sim_cmd  = self.cmd_part + ['-c', contig_id] + ['-n', str(self.count_dict[contig_id])]
    read_gen = LockedIterator(StreamHTSIM(sim_cmd=sim_cmd, pair_end=self.pair_end)) # only output 1 header for -c TODO
    var_contig, sim_data= next(read_gen)                                    # the first element of generator is variants
    self.current_contig = var_contig                                        # update the profiles
    self.pos_map, self.meth_arr, _ = self.meth_db.load_contig(var_contig)   # [pos_map, meth_arr, status]
    self.variant_profile= self.meth_set.set_var_meth(var_contig, sim_data)  # a dict, can be empty

    with ThreadPoolExecutor(max_workers=self.n_threads) as executor:
        if self.pair_end:                                                       # what if read_gen is empty at very beginning
            job_arr = [executor.submit(self.test, read_pair) for _, read_pair in read_gen]
        else:
            job_arr = [executor.submit(self.test, read_pair) for _, read_pair in read_gen]

            
        for job in as_completed(job_arr):
            try:
                data = job.result()
            except Exception as exc:
                print('generated an exception: %s' % (exc))
            else:
                pass

Simulating whole genome reads:
Reference genome file: /home/wbguo/iproject/BSReadSim/test/data/ref/BSB_test.fa
[main] Calculating the total length and effective length of the reference sequences...
[main] Contig chr10 specified, contig length: 423500, effective length: 423500
[main] No VCF input, will generate SNP randomly if mutation rate is nonzero
[htsim] seed = 1677728581
[sim_core] contig 'chr10': simulate 10794 reads...
[sim_core] Generated 10794 read pairs, with 1153 contain SNP, 262 contain INDEL
Simulating whole genome reads:
Reference genome file: /home/wbguo/iproject/BSReadSim/test/data/ref/BSB_test.fa
[main] Calculating the total length and effective length of the reference sequences...
[main] Contig chr11 specified, contig length: 424000, effective length: 424000
[main] No VCF input, will generate SNP randomly if mutation rate is nonzero
[htsim] seed = 1677728585
[sim_core] contig 'chr11': simulate 10808 reads...
[sim_core] Generated 10808 read pairs, with 1077 contain SNP